# NAS - Optuna Study

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

# TODO:

IMPLEMENT PANDAS FOR THE DATASET INSTEAD OF NUMPY

ADD LOAD STUDY FUNCTIONALITY

## 1. Imports

In [ ]:
# --------------------------- Standard Libraries ---------------------------- #
import os
import gc
import math
import time
import psutil
import subprocess
import traceback
from threading import Thread

# -------------------------------- Annotations ------------------------------- #
from typing import List, Optional, Tuple

# ------------------------- Data Processing Libraries ----------------------- #
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
import pandas as pd

# ----------------------- TensorFlow and Keras Modules ---------------------- #
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import AdamW, SGD, RMSprop, Adam, Nadam
from tensorflow.keras.backend import clear_session
from tensorflow.keras import layers, Model

tf.get_logger().setLevel('ERROR')

# ---------------------------- Scikit-learn Modules ------------------------- #
from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    RobustScaler,
    QuantileTransformer,
    PowerTransformer,
)
from sklearn.model_selection import train_test_split

# ---------------------------------- Optuna ---------------------------------- #
import optuna
# import optunahub # For the AutoSampler

## 2. Utility Functions Definitions

### 2.1. GPU

In [ ]:
def get_gpu_info():
    """
    Retrieves and prints detailed GPU information including TensorFlow,
    CUDA, cuDNN versions, number of GPUs, and memory details.
    """
    # Display TensorFlow version
    print(f"TensorFlow Version: {tf.__version__}")

    # Check if TensorFlow is built with CUDA support and retrieve build info
    if tf.test.is_built_with_cuda():
        build_info = tf.sysconfig.get_build_info()
        print(f"TensorFlow is built with CUDA support")
        print(f"CUDA Version: {build_info['cuda_version']}")
        print(f"cuDNN Version: {build_info['cudnn_version']}")
    else:
        print("Running on CPU (No CUDA support detected)")

    # Detect available GPUs
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        print(f"\nNumber of GPUs detected: {len(gpus)}")
        print(f"Available GPU(s): {[gpu.name for gpu in gpus]}\n")
        tf.test.gpu_device_name()
    else:
        print("No GPUs found")
        print("Running on CPU")

### 2.2. Folder and Files

In [ ]:
def create_run_directory(prefix: str, base_dir: str = "runs") -> str:
    """
    Creates a new directory for storing training logs, checkpoints, and plots.
    The directory name is based on the next available number.

    Args:
        prefix (str): Prefix for the run directory.
        base_dir (str): Base directory for storing training runs. Defaults to "runs".

    Returns:
        str: Path to the created run directory.
    """
    os.makedirs(base_dir, exist_ok=True)  # Ensure the base directory exists

    # Find the next available run number
    existing_dirs = [d for d in os.listdir(base_dir) if d.startswith(prefix) and d[len(prefix):].isdigit()]
    next_run_number = max([int(d[len(prefix):]) for d in existing_dirs] + [0]) + 1
    run_dir = os.path.join(base_dir, f"{prefix}{next_run_number}")
    os.makedirs(run_dir, exist_ok=True)  # Create the run directory

    return run_dir


def save_trial_params_to_file(filepath: str, params: dict, **kwargs) -> None:
    """
    Saves the parameters of a trial along with additional information to a file.

    Args:
        filepath (str): Path to the file where parameters will be saved.
        params (dict): Parameters of the trial to save.
        **kwargs: Additional information to include in the file (e.g., rank, trial ID, loss).
    """
    with open(filepath, "w") as txt_file:
        # Write additional info
        for key, value in kwargs.items():
            txt_file.write(f"{key}: {value}\n")
        
        # Write parameters
        txt_file.write("Parameters:\n")
        for param_name, param_value in params.items():
            txt_file.write(f"  {param_name}: {param_value}\n")

### 2.3. Plotting

In [ ]:
def plot_scientific(
    x: List[float],
    y_datasets: List[List[float]],
    labels: Optional[List[str]] = None,
    x_label: str = "X-axis",
    y_label: str = "Y-axis",
    title: str = "Scientific Plot",
    x_integer: bool = False,
    y_log: bool = False,
    markers: Optional[List[str]] = None,
    line_styles: Optional[List[str]] = None,
    legend_loc: str = "best",
    grid: bool = True,
    xlim: Optional[Tuple[float, float]] = None,
    ylim: Optional[Tuple[float, float]] = None,
    save_path: Optional[str] = None,
    dpi: int = 300,
) -> None:
    """
    Plots multiple Y datasets against a shared X-axis with scientific paper styling.

    Args:
        x (List[float]): List of X-axis values.
        y_datasets (List[List[float]]): List of lists containing Y-axis datasets.
        labels (Optional[List[str]]): Labels for each dataset for the legend.
        x_label (str): Label for the X-axis. Default is "X-axis".
        y_label (str): Label for the Y-axis. Default is "Y-axis".
        title (str): Title of the plot. Default is "Scientific Plot".
        x_integer (bool): Force X-axis to display only integer values. Default is False.
        y_log (bool): Use logarithmic scale for the Y-axis. Default is False.
        markers (Optional[List[str]]): List of marker styles for each dataset.
        line_styles (Optional[List[str]]): List of line styles for each dataset.
        legend_loc (str): Location of the legend. Default is "best".
        grid (bool): Whether to display a grid. Default is True.
        xlim (Optional[Tuple[float, float]]): Limits for the X-axis as (min, max). Default is None.
        ylim (Optional[Tuple[float, float]]): Limits for the Y-axis as (min, max). Default is None.
        save_path (Optional[str]): Path to save the plot as a file. Default is None.
        dpi (int): Resolution of the saved plot in dots per inch. Default is 300.

    Returns:
        None: Displays the plot and optionally saves it as an image.
    """
    # Validate input dimensions
    if any(len(y) != len(x) for y in y_datasets):
        raise ValueError("All Y datasets must have the same length as the X dataset.")

    # Initialize the plot
    plt.figure(figsize=(8, 6))

    # Plot each dataset
    for i, y in enumerate(y_datasets):
        label = labels[i] if labels and i < len(labels) else f"Dataset {i + 1}"
        marker = markers[i] if markers and i < len(markers) else "o"
        line_style = line_styles[i] if line_styles and i < len(line_styles) else "-"
        plt.plot(x, y, label=label, marker=marker, linestyle=line_style)

    # Configure axes and title
    plt.xlabel(x_label, fontsize=12)
    plt.ylabel(y_label, fontsize=12)
    plt.title(title, fontsize=14, weight="bold")
    plt.legend(loc=legend_loc, fontsize=10)

    # Configure axis limits
    if xlim:
        plt.xlim(xlim)
    if ylim:
        plt.ylim(ylim)

    # Configure X-axis for integers only
    if x_integer:
        plt.xticks(ticks=range(int(min(x)), int(max(x)) + 1))

    # Enable logarithmic scale for Y-axis if requested
    if y_log:
        plt.yscale("log")

    # Add grid if requested
    if grid:
        plt.grid(visible=True, linestyle="--", linewidth=0.5, alpha=0.7)

    # Final styling
    plt.tight_layout()

    # Save plot if path is provided
    if save_path:
        plt.savefig(save_path, dpi=dpi, format="png")

    # Show plot
    plt.show()

### 2.4. Logging

In [ ]:
def log_resources(log_dir: str, interval: int = 5, **kwargs) -> None:
    """
    Logs selected resources (CPU, RAM, GPU, CUDA, TensorFlow) at regular intervals.

    Args:
        log_dir (str): Directory to save log files.
        interval (int): Time interval between logs (seconds). Default is 5.
        kwargs: Resource options as boolean. Supported options: 
                "cpu", "ram", "gpu", "cuda", "tensorflow".
    """
    os.makedirs(log_dir, exist_ok=True)

    def log_cpu():
        log_path = os.path.join(log_dir, "cpu_usage_log.csv")
        with open(log_path, "w") as f:
            f.write("Timestamp,CPU_Usage(%),Per-Core_Usage(%)\n")
            while True:
                try:
                    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
                    cpu_usage = psutil.cpu_percent()
                    per_core_usage = psutil.cpu_percent(percpu=True)
                    f.write(f"{timestamp},{cpu_usage},{','.join(map(str, per_core_usage))}\n")
                    f.flush()
                    time.sleep(interval)
                except Exception as e:
                    f.write(f"Error: {e}\n")
                    break

    def log_ram():
        log_path = os.path.join(log_dir, "ram_usage_log.csv")
        with open(log_path, "w") as f:
            f.write("Timestamp,Total(MB),Used(MB),Free(MB)\n")
            while True:
                try:
                    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
                    mem = psutil.virtual_memory()
                    total = mem.total / (1024 ** 2)
                    used = mem.used / (1024 ** 2)
                    free = mem.available / (1024 ** 2)
                    f.write(f"{timestamp},{total:.2f},{used:.2f},{free:.2f}\n")
                    f.flush()
                    time.sleep(interval)
                except Exception as e:
                    f.write(f"Error: {e}\n")
                    break

    def log_gpu():
        log_path = os.path.join(log_dir, "gpu_usage_log.csv")
        with open(log_path, "w") as f:
            f.write("Timestamp,GPU_ID,Memory_Used(MB),Memory_Total(MB),GPU_Utilization(%)\n")
            while True:
                try:
                    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
                    gpu_stats = subprocess.run(
                        ["nvidia-smi", "--query-gpu=index,memory.used,memory.total,utilization.gpu",
                         "--format=csv,noheader,nounits"],
                        capture_output=True,
                        text=True,
                    ).stdout.strip()
                    
                    for line in gpu_stats.split("\n"):
                        gpu_id, mem_used, mem_total, util = map(int, line.split(","))
                        f.write(f"{timestamp},{gpu_id},{mem_used},{mem_total},{util}\n")
                    f.flush()
                    time.sleep(interval)
                except Exception as e:
                    f.write(f"Error: {e}\n")
                    break

    def log_cuda():
        log_path = os.path.join(log_dir, "cuda_usage_log.csv")
        with open(log_path, "w") as f:
            f.write("Timestamp,Process_Memory_Used(MB)\n")
            while True:
                try:
                    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
                    cuda_mem_stats = subprocess.run(
                        ["nvidia-smi", "--query-compute-apps=used_memory", "--format=csv,noheader,nounits"],
                        capture_output=True,
                        text=True,
                    ).stdout.strip()
                    if cuda_mem_stats:
                        f.write(f"{timestamp},{cuda_mem_stats} MB\n")
                    else:
                        f.write(f"{timestamp},0 MB\n")
                    f.flush()
                    time.sleep(interval)
                except Exception as e:
                    f.write(f"Error: {e}\n")
                    break

    def log_tensorflow():
        log_path = os.path.join(log_dir, "tensorflow_usage_log.csv")
        os.makedirs(log_dir, exist_ok=True)
        with open(log_path, "w") as f:
            f.write("Timestamp,Device,Memory_Allocated(MB),Memory_Peak(MB)\n")
            while True:
                try:
                    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
                    gpus = tf.config.experimental.list_physical_devices("GPU")
                    for gpu in gpus:
                        device_name = gpu.name  # The correct device name
                        memory_info = tf.config.experimental.get_memory_info("GPU:0")
                        allocated_memory = memory_info["current"] / (1024 ** 2)  # Convert to MB
                        peak_memory = memory_info["peak"] / (1024 ** 2)  # Convert to MB
                        f.write(f"{timestamp},{device_name},{allocated_memory:.2f},{peak_memory:.2f}\n")
                    f.flush()
                    time.sleep(interval)
                except Exception as e:
                    f.write(f"Error: {e}\n")
                    break

    # Start logging threads based on user selection
    if kwargs.get("cpu", False):
        Thread(target=log_cpu, daemon=True).start()
    if kwargs.get("ram", False):
        Thread(target=log_ram, daemon=True).start()
    if kwargs.get("gpu", False):
        Thread(target=log_gpu, daemon=True).start()
    if kwargs.get("cuda", False):
        Thread(target=log_cuda, daemon=True).start()
    if kwargs.get("tensorflow", False):
        Thread(target=log_tensorflow, daemon=True).start()

## 3. Setup and Configuration

### 3.1. GPU Management

In [ ]:
# Specify GPU to use (e.g., GPU 0)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
get_gpu_info()

### 3.2. Folder and File Management

In [ ]:
# Create the base directory for the current run:
RUN_DIR = create_run_directory(prefix="nas_dnn_")

LOG_DIR = os.path.join(RUN_DIR, "logs")

os.makedirs(LOG_DIR, exist_ok=True)


## 4. Constants and Hyperparameters

In [ ]:
# ---------------------------- Dataset Parameters ---------------------------- #
TOTAL_NUM_PORTS = 144  # Total number of ports in the dataset

# ---------------------------- Training Parameters --------------------------- #
NUM_TRIALS = 100  # Number of hyperparameter optimization trials
EPOCHS = 50  # Number of epochs to train
TOP_K = 5  # Number of top trials to save

# Runs with:
# NUM_TRIALS = 100
# EPOCHS = 50

observed_ports_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# ------------------------- Randomization Parameters ------------------------- #
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 5. Data Loading and Preprocessing

### 5.1. Data Loading

In [ ]:
# --------------------- Load the dataset in matlab format -------------------- #
# TODO: remake this and utils codes to use pandas instead
dataset = (scipy.io.loadmat('data/shadowed_rayleigh/SNR_events.mat')['SNR_events']).T

print(f"Original dataset shape: {dataset.shape}")

# ---------------------------- Data Configuration ---------------------------- #
# Subsample data
dataset = dataset[:int(0.01 * dataset.shape[0]), :] # 1.0 means full dataset

# Changing the scale of the data
dataset = np.log10(dataset)
#! Warning: this makes the numbers much closer, which may make training harder

print(f"Shape of the data after configuration: {dataset.shape}\n")

## 6. Implementation Functions Definitions

In [ ]:
def get_observed_ports(sinr_data, num_observed_ports, total_ports):
    """
    Extracts SINR values for the specified number of observed ports.

    The function selects a subset of SINR data by identifying equally spaced ports based on the
    number of observed ports specified. It returns the SINR values for these observed ports and
    their corresponding indices.

    Args:
        sinr_data (numpy.ndarray): A 2D array where each row represents an observation and each column
                                   represents a port with its corresponding SINR values.
        num_observed_ports (int): The number of observed ports to select from the SINR data.
        total_ports (int): The total number of ports in the SINR data.

    Returns:
        observed_sinr (numpy.ndarray): A 2D array containing the SINR values for the observed ports.
        observed_indices (numpy.ndarray): A 1D array of the indices corresponding to the observed ports.
    """
    observed_indices = np.linspace(0, total_ports - 1, num_observed_ports, dtype=int)
    observed_sinr = sinr_data[:, observed_indices]

    return observed_sinr, observed_indices

In [ ]:
# Test the get_observed_ports function
observed_sinr, observed_indices = get_observed_ports(
    dataset, num_observed_ports=10, total_ports=TOTAL_NUM_PORTS
)
print(f"Observed SINR shape: {observed_sinr.shape}")
print(f"Observed port indices: {observed_indices}")

## 7. Optuna Study

### 7.1. Objective Function Definition

In [ ]:
def objective(
    trial: optuna.Trial,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    input_shape: int,
    output_units: int,
    checkpoint_path: str,
    model_path: str,
    fig_dir: str,
) -> float:
    """
    Objective function for optimizing a Neural Network.

    Args:
        trial (optuna.Trial): A single trial of the optimization process.
        X_train (np.ndarray): The input data for training the DNN.
        y_train (np.ndarray): The target data for training the DNN.
        X_test (np.ndarray): The input data for testing the DNN.
        y_test (np.ndarray): The target data for testing the DNN.
        input_shape (int): Shape of the input data.
        output_units (int): Number of output units.
        checkpoint_path (str): Path to a checkpoint file.
        model_path (str): Path to save the model.
        fig_dir (str): Directory to save the figures.

    Returns:
        float: The best validation loss achieved during this trial.
    """

    # -------------------------------- Input Layer ------------------------------- #
    inputs = layers.Input(shape=(input_shape,))  # Observed ports as input
    x = inputs

    # ------------------------- Funnel-shaped DNN Layers ------------------------- #
    # ? TIP: Could let optuna find the units for layer 0, then funnel the next layers
    # ? by dividing the units by a fixed proportion, e.g., previous_units // 2
    
    dnn_layers = trial.suggest_int("dnn_layers", 1, 10)  # Total layers in the funnel
    units_layer_0 = trial.suggest_int("dnn_units_layer_0", 128, 1024, step=128)

    for i in range(dnn_layers):
        # Next layer will have fewer units than the previous layer
        units = (
            units_layer_0
            if i == 0
            else trial.suggest_int(
                f"dnn_units_layer_{i}",
                max(16, math.ceil(units_previous_layer // 2 / 16) * 16),
                math.floor(units_previous_layer / 16) * 16,
                step=16,
            )
        )
        units_previous_layer = units

        activation: str = trial.suggest_categorical(
            f"dnn_activation_layer_{i}",
            [
                "relu",  # [0, ∞)
                "tanh",  # (-1, 1)
                "sigmoid",  # [0, 1]
                # "linear",  # (-∞, ∞)
                # "elu",  # (-x, ∞), default x=1
                # "selu",  # (-∞, ∞)
                # "swish",  # (-∞, ∞)
                # "softplus",  # (0, ∞)
            ],
        )

        # ------------------------------- The DNN layer ------------------------------ #
        x = layers.Dense(
            units=units,
            activation=activation,
            # L2 class: loss = l2 * reduce_sum(square(x))
            kernel_regularizer=tf.keras.regularizers.l2(
                # trial.suggest_float(f"kernel_regularizer_layer_{i}", 0.0, 0.1)
            ),  # Apply a penalty on the layer's kernel
            bias_regularizer=tf.keras.regularizers.l2(
                # trial.suggest_float(f"bias_regularizer_layer_{i}", 0.0, 0.1)
            ),  # Apply a penalty on the layer's bias
            activity_regularizer=tf.keras.regularizers.l2(
                # trial.suggest_float(f"activity_regularizer_layer_{i}", 0.0, 0.1),
            ),  # Apply a penalty on the layer's output
        )(x)

        # ------------------------- Use Batch Normalization? ------------------------- #
        if trial.suggest_categorical(f"dnn_use_batch_norm_layer_{i}", [True, False]):
            x = layers.BatchNormalization()(x)

        # -------------------------- Use Dropout? How much? -------------------------- #
        x = layers.Dropout(trial.suggest_categorical(f"dnn_dropout_layer_{i}", [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]))(x)

    # ------------------------------- Output layer ------------------------------- #
    # Generate the predicted SINR value for each port, REGRESSION
    outputs = layers.Dense(
        output_units,  # Output: (batch_size, TOTAL_NUM_PORTS)
        activation="linear",  # (-∞, ∞)
    )(x)

    # --------------------------------- Optimizer -------------------------------- #
    optimizer_name = trial.suggest_categorical(
        "optimizer",
        [
            "AdamW",
            "SGD",
            "RMSprop",
            # "Adam",
            # "Nadam",
        ],
    )
    learning_rate = trial.suggest_float("learning_rate", 0.0001, 0.01)

    if optimizer_name == "AdamW":
        optimizer = AdamW(learning_rate=learning_rate)
    elif optimizer_name == "SGD":
        optimizer = SGD(
            learning_rate=learning_rate,
            # momentum=trial.suggest_float("momentum", 0.0, 0.99),
        )
    elif optimizer_name == "RMSprop":
        optimizer = RMSprop(learning_rate=learning_rate)
    elif optimizer_name == "Adam":
        optimizer = Adam(learning_rate=learning_rate)
    elif optimizer_name == "Nadam":
        optimizer = Nadam(learning_rate=learning_rate)

    # ----------------------- Create and Compile the Model ----------------------- #
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=optimizer, loss="mse")

    # --------------------------------- Callbacks -------------------------------- #
    early_stopping = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, mode="min")
    reduce_lr = ReduceLROnPlateau(monitor="val_loss", patience=5)
    checkpoint_callback = ModelCheckpoint(
        filepath=os.path.join(checkpoint_path, f"trial_{trial.number}_best_model.weights.h5"),
        save_weights_only=True,  # Save only the weights
        monitor="val_loss",  # Monitor validation loss
        mode="min",
        save_best_only=True,  # Save only the best model
        verbose=0,
    )

    # ---------------------------- Data preprocessing ---------------------------- #
    scaler_name = trial.suggest_categorical(
        "scaler",
        [
            "StandardScaler",  # Gaussian-like distributed data. Mean-centered scaling with unit variance is needed.
            # "RobustScaler",  # Data with significant outliers, as it uses the median and IQR for scaling.
            # "QuantileTransformer",  # Skewed data when you want a uniform or normal distribution via non-linear transformation.
            # "PowerTransformer",  # Heavily skewed data to stabilize variance and achieve Gaussian-like distributions.
            "MinMaxScaler_0_1",  # Data with no significant outliers and features bounded to a range.
            "MinMaxScaler_-1_1",  # Data with no significant outliers and features bounded to a range.
        ],
    )

    # Initialize the selected scaler
    if scaler_name == "StandardScaler":
        scaler = StandardScaler()
    elif scaler_name == "RobustScaler":
        scaler = RobustScaler()
    elif scaler_name == "QuantileTransformer":
        scaler = QuantileTransformer(output_distribution="normal")
    elif scaler_name == "PowerTransformer":
        scaler = PowerTransformer(method="yeo-johnson")
    elif scaler_name == "MinMaxScaler_0_1":
        scaler = MinMaxScaler(feature_range=(0, 1))
    elif scaler_name == "MinMaxScaler_-1_1":
        scaler = MinMaxScaler(feature_range=(-1, 1))

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # ------------------------------ Train the model ----------------------------- #
    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=trial.suggest_categorical("batch_size", [32, 64, 128, 256, 512, 1024]),
        validation_data=(X_test, y_test),
        callbacks=[early_stopping, reduce_lr, checkpoint_callback],
        shuffle=True,
        verbose=0,
    )
    
    model.save(os.path.join(model_path, f"trial_{trial.number}_model.keras"))
 
    plot_scientific(
        x=range(1, len(history.history["loss"]) + 1),
        y_datasets=[history.history["loss"], history.history["val_loss"]],
        labels=["Training MSE", "Validation MSE"],
        x_label="Epoch",
        y_label="Mean Squared Error (MSE)",
        title=f"Training and Validation MSE with {input_shape} Observed Ports",
        markers=["o", "x"],
        line_styles=["-", "--"],
        ylim=(
            0,
            max(max(history.history["loss"]), max(history.history["val_loss"])) * 1.05,
        ),
        grid=True,
        x_integer=True,
        save_path=os.path.join(fig_dir, f"trial_{trial.number}_mse_plot.png"),
    )

    del model  # Delete the model to free up memory
    clear_session()  # Clean up TensorFlow variables
    gc.collect()  # Collect garbage

    return min(history.history["val_loss"])

### 7.2 Code Health Info

In [ ]:
log_resources(
    log_dir=LOG_DIR,
    interval=5,
    cpu=True,
    ram=True,
    gpu=True,
    cuda=True,
    tensorflow=True,
)

In [ ]:
# ---------------------------- Kernel life monitor --------------------------- #
#! Remove this block if you don't want to use it
try:
    pid_path = os.path.join(RUN_DIR, "kernel_pid.txt")
    pid = os.getpid()
    with open(pid_path, "w") as f:
        f.write(str(pid))
    print(f"[INFO] Kernel PID saved to {pid_path}: {pid}")
    print(f"Call the monitor script: python _monitor_kernel_life.py --pid_path {pid_path} --interval 10\n")

    # from _monitor_kernel_life import start_monitoring
    # start_monitoring(pid_path=pid_path, check_interval=5)
except Exception as e:
    print("[ERROR] Kernel monitoring failed!")
    print(e)
    pass

### 7.3. Study

The **sampler** in Optuna determines how the next set of hyperparameters is chosen for evaluation during the optimization process.

You can also use AutoSampler to automatically select the best sampler for your optimization problem, refer to the [AutoSampler](https://medium.com/optuna/autosampler-automatic-selection-of-optimization-algorithms-in-optuna-1443875fd8f9) and [Optuna documentation](https://hub.optuna.org/samplers/auto_sampler/).

For more detailed information on each sampler, refer to the [Optuna documentation](https://optuna.readthedocs.io/en/stable/reference/samplers/index.html). 

In [ ]:
for n in observed_ports_list:
    print(f"Optimizing for {n} observed ports...")

    # ------------------------------- Storage paths ------------------------------ #
    study_name = os.path.join(RUN_DIR, f"study_for_{n}_observed_ports")
    os.makedirs(study_name, exist_ok=True)

    args_dir = os.path.join(study_name, "args")
    os.makedirs(args_dir, exist_ok=True)

    fig_dir = os.path.join(study_name, "figures")
    os.makedirs(fig_dir, exist_ok=True)

    storage_path = f"sqlite:///{os.path.join(study_name, 'optuna_study.db')}"
    config_save_path = os.path.join(args_dir, f"best_args_observed_ports_{n}.yaml")
    checkpoint_path = os.path.join(study_name, "weights")

    model_path = os.path.join(study_name, "models")
    os.makedirs(model_path, exist_ok=True)

    print(f"Initializing study with name '{study_name}'...")

    # ---------------------------- Data Preprocessing --------------------------- #
    observed_ports, _ = get_observed_ports(dataset, num_observed_ports=n, total_ports=TOTAL_NUM_PORTS)
    X_train, X_test, y_train, y_test = train_test_split(
        observed_ports, dataset, test_size=0.2, random_state=SEED
    )

    # ---------------------------------- Pruners --------------------------------- #
    pruner = optuna.pruners.SuccessiveHalvingPruner()  # Judge whether the trial should be pruned

    # ----------------------------------- Study ---------------------------------- #
    study = optuna.create_study(
        study_name=study_name,
        storage=storage_path,
        direction="minimize",
        pruner=pruner,
        load_if_exists=False,
        
        # If you want to change the sampler:
        # sampler=optunahub.load_module(package="samplers/auto_sampler").AutoSampler()
    )

    study.optimize(
        lambda trial: objective(
            trial,
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            input_shape=n,
            output_units=TOTAL_NUM_PORTS,
            checkpoint_path=checkpoint_path,
            model_path=model_path,
            fig_dir=fig_dir,
        ),
        n_trials=NUM_TRIALS,
        catch=(Exception,),
    )

    # ----------------------------- Save Top-K Trials ---------------------------- #
    valid_trials = [t for t in study.trials if t.value is not None]
    sorted_trials = sorted(valid_trials, key=lambda t: t.value)[:TOP_K]

    for rank, trial in enumerate(sorted_trials):
        trial_id = trial.number
        trial_params = trial.params
        trial_loss = trial.value

        # Save the best configuration using save_trial_params_to_file
        save_trial_params_to_file(
            filepath=os.path.join(args_dir, f"top_{rank + 1}_trial_observed_ports_{n}.txt"),
            params=trial_params,
            rank=rank + 1,
            trial_id=trial_id,
            loss=trial_loss,
            sampler=study.sampler.__class__.__name__,
        )

    # ------------------------ Clean-Up Non-Top Trials --------------------------- #
    all_trial_ids = {t.number for t in study.trials}
    top_trial_ids = {t.number for t in sorted_trials}

    # Delete non-top weights
    for trial_id in all_trial_ids - top_trial_ids:
        weights_path = os.path.join(checkpoint_path, f"trial_{trial_id}_best_model.weights.h5")
        if os.path.exists(weights_path):
            os.remove(weights_path)

    # Delete non-top models
    for trial_id in all_trial_ids - top_trial_ids:
        model_file_path = os.path.join(model_path, f"trial_{trial_id}_model.keras")
        if os.path.exists(model_file_path):
            os.remove(model_file_path)

    # Delete non-top mse plots
    for trial_id in all_trial_ids - top_trial_ids:
        mse_plot_path = os.path.join(fig_dir, f"trial_{trial_id}_mse_plot.png")
        if os.path.exists(mse_plot_path):
            os.remove(mse_plot_path)

    print(f"Saved top-{TOP_K} trials for {n} observed ports.\n")
    print("--------------------------------------\n\n")

    # ----------------------- Email api to notify the user ----------------------- #
    #! Remove this block if you don't want to send an email
    try:
        from utils.email_api import send_email

        # Message that the study for n observed ports is complete
        email_subject = f"Model Study for {n} Observed Ports Complete"
        email_body = f"""
        <html>
            <body style="font-family: Arial, sans-serif; line-height: 1.6; color: #333; background-color: #f9f9f9; padding: 20px;">
            <div style="max-width: 600px; margin: auto; background: #fff; padding: 20px; border: 1px solid #ddd; border-radius: 8px;">
                <h2 style="color: #0056b3; text-align: center;">🎉 Study Complete</h2>
                <p style="font-size: 16px; color: #444;">
                <strong>Dear User,</strong>
                </p>
                <p style="font-size: 18px; color: #333;">
                Your study for <strong style="color: #0056b3;">{n} Observed Ports</strong> has successfully completed!
                </p>
                <p style="font-size: 16px; color: #555;">
                The best trial achieved a loss value of <strong style="color: #0056b3;">{study.best_trial.value}</strong>.
                </p>
                <p style="text-align: center; font-size: 16px;">
                <strong style="color: #28a745;">✔️ Study Status:</strong> <span style="color: #0056b3;">Completed</span>
                </p>
                <footer style="margin-top: 20px; text-align: center; font-size: 14px; color: #888;">
                <p>Best regards,</p>
                <p><strong>The Optimization Team</strong></p>
                </footer>
            </div>
            </body>
        </html>
        """
        send_email(
            subject=email_subject,
            body=email_body,
            recipients_file="./json/recipients.json",
            credentials_file="./json/credentials.json",
            text_type="html",
        )
    except Exception as e:
        print(f"[ERROR] Failed to send email: {e}")
        traceback.print_exc()
        pass